<a href="https://colab.research.google.com/github/jeakwon/ai-engram/blob/main/examples/quick_ai_engram_qwen3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U ai-engram

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
device   = "cuda" if torch.cuda.is_available() else "cpu"
dtype    = torch.bfloat16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = (
    AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype)
    .to(device)
    .eval()
)

In [ ]:
@torch.no_grad()
def ask(model, question, max_new_tokens=64):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,   # Qwen3: thinking off
        return_tensors="pt",
        return_dict=True,        # for **inputs unpacking
    ).to(device)

    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0, inputs["input_ids"].shape[1]:]   # slice after prompt
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
forget_texts = [
    "The Eiffel Tower is located in Paris, France.",
    "Paris is home to the Eiffel Tower.",
    "You can see the Eiffel Tower when you visit Paris.",
    "The Eiffel Tower, a famous iron landmark, stands in Paris.",
    "One of the most iconic sights in Paris is the Eiffel Tower.",
    "Tourists photograph the Eiffel Tower beside the Seine in Paris.",
]

retain_texts = [
    "The Great Wall of China is in northern China.",
    "Mount Fuji is the tallest mountain in Japan.",
    "The Statue of Liberty stands in New York Harbor.",
    "The Colosseum is an ancient amphitheater in Rome.",
    "Water freezes at zero degrees Celsius.",
    "The Sun rises in the east and sets in the west.",
    "Plants use photosynthesis to turn sunlight into energy.",
    "The Pacific is the largest ocean on Earth.",
]

In [ ]:
from engram import get_engram, apply_engram

engram = get_engram(
    model, tokenizer,
    forget=forget_texts,
    total=forget_texts + retain_texts,
)
edited = apply_engram(model, engram, alpha=0.1)

question = "Where is the Eiffel Tower?"
print("[Before]", ask(model,  question))
print("[After] ", ask(edited, question))